# VASP: Adsorção

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares) 

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## Criando a molécula de H2O e otimizando sua geometria

Criando a molécula de água (H2O) utilizando o ASE

In [ ]:
from ase.build import molecule

molecule = molecule('H2O')
molecule.center(vacuum=4.0) # caixa com 4 Angstroms de vácuo 
molecule.pbc = True # condição de contorno periódica


from ase.calculators.vasp import Vasp

calc = Vasp(directory='adsorcao/h2o',
            xc="PBE",
            encut=350,
            kpts=[1, 1, 1],gamma=True,                  # k-points
            ibrion=2, # CG ionic relax
            isif=0, # relaxa somente os átomos, mantendo a célula fixa
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-2, # critério de convergência para relaxação (forças)
            ivdw=12,                  # Adicionando correção de van der Waals D3(BJ) (molécula grande)
            atoms=molecule)

E_h2o = molecule.get_potential_energy()

print(f'Energia total da molécula de água: {E_h2o:.3f} eV')

## Criando um cristal de Ag e otimizando sua geometria

In [ ]:
from ase.build import bulk

crystal = bulk("Ag", crystalstructure="fcc", a=4.0, cubic=True)

calc = Vasp(directory='adsorcao/Ag',
            xc='PBE',   # funcional GGA
            encut=350,  # safe default for PAW-PBE sets
            kpts=[1,1,1],gamma=True,                  # k-points
            ibrion=2, # CG ionic relax
            isif=7, # relaxa somente a célula
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-2, # critério de convergência para relaxação (forças)
            atoms=crystal
)

E_crystal = crystal.get_potential_energy()
print(f'Energia total do cristal: {E_crystal:.3f} eV')

print("Nova estrutura do cristal (Ang):")
print(crystal.get_cell())

## Criando um slab de Ag(111)

Criando Ad(111) como superfície adsorvente

In [ ]:
from ase.visualize import view
from ase.build import fcc111

slab = fcc111("Ag", size=(2,2,3), a=4.5, vacuum=10.0,periodic=True)

view(slab, viewer='x3d')

Calculando energia do slab

In [ ]:
calc = Vasp(directory='adsorcao/Ag111',
            xc='PBE',
            kpts=[1,1,1],  # specifies k-points
            encut=350,
            ibrion=2, # CG ionic relax
            isif=7, # relaxa somente a célula
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-2, # critério de convergência para relaxação (forças)
            lvtot=True,  # keep LOCPOT for post-processing
            atoms=slab)

E_slab = slab.get_potential_energy()
print("Energia do slab Ag(111) = {:.3f} eV".format(E_slab))

## Criando sistema de adorção = slab + molecula

Adicionando molécula como adsorvato

In [ ]:
adsorvato = molecule.copy()

# centralizando a molécula na superfície do slab
adsorvato.center(about=slab.get_positions()[-9:].mean(axis=0)) 
adsorvato.translate([0.0,0.0,2.5])  # altura de 2.5 Å do slab

# criando sistema de adorção = slab + adsorvato
ads_system = slab.copy()
ads_system += adsorvato

Fixando os átomos de Ag 

In [ ]:
from ase.constraints import FixAtoms

constraint = FixAtoms(mask=[atom.symbol=='Ag' for atom in ads_system])
ads_system.set_constraint(constraint)

In [ ]:
view(ads_system, viewer='x3d')

In [ ]:
# Vendo a posição de cada átomo no sistema 
for id, atom in enumerate(ads_system):
    print(f"Atom ID: {id}, Atom: {atom.symbol}, Position: {atom.position}")

Calculando a distância inicial entre a molécula e superfície

In [ ]:
z_slab_top = max([atom.position[2] for atom in ads_system if atom.symbol=='Ag'])

z_oxygen_position = [atom.position[2] for atom in ads_system if atom.symbol=='O']

distance = z_oxygen_position[0] - z_slab_top

print("Distância do oxigênio até o topo do slab: {:.3f} Å".format(distance))

Criando calculadora do VASP para otimizar posição do adsorvato

In [ ]:
calc = Vasp(directory='adsorcao/h2o-Ag111-center',
            xc='PBE',
            kpts=[1,1,1],  
            encut=350,
            ibrion=2, # CG ionic relax
            isif=7, # relaxa somente a célula
            nsw=50, # número máximo de passos de relaxação
            ediffg=-1e-2, # critério de convergência para relaxação (forças)
            ivdw=12,    # Adicionando correção de van der Waals D3(BJ) 
            atoms=ads_system)

E_ads_center = ads_system.get_potential_energy()

print("Energia do sistema Ag(111) + H2O (center) = {:.3f} eV".format(E_ads_center))

In [ ]:
view(ads_system, viewer='x3d')

Calculando energia de adsorção

In [ ]:
E_adsorcao = E_ads_center - E_h2o - E_slab
print(f'Energia de adsorção: {E_adsorcao:.3f} eV')